# Beyond - Linear Attention

This notebook proves that the parallel reference and recurrent forms agree, benchmarks fixed-state decoding against a growing KV cache, trains a StoryLM-1M full/linear/hybrid comparison on TinyStories, and measures recall without assuming the result. The optional final exercise repeats the language-model comparison at StoryLM-5M scale.

1. Read the lesson page (`docs/beyond/linear-attention.md`).
2. Open this notebook with `./notebook.sh linear-attention`.
3. Answer the `Question:` / `Answer:` cells below.
4. When you're ready, ask a coding agent to grade your notebook.

Partial work is fine. Blank `Answer: ""` strings are skipped, not counted wrong. If you'd like a hint instead of a grade, write the request inline in the answer string and the agent will tutor first.

In [ ]:
import time

import torch
import matplotlib.pyplot as plt

from g2c.attention import MultiHeadAttention
from g2c.linear_attention import HybridTransformerLM, LinearAttention

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"device: {device}")

## Exercise 1 — Prove the equivalence, then benchmark decoding

One computation, two forms. First prove the readable parallel reference agrees with recurrent stepping. Then compare like with like at inference: one cached full-attention decoding step against one linear recurrent step over growing histories. The benchmark measures the course's deliberately simple cache implementation, including its concatenate-on-append overhead, and reports what this machine actually does.

The training story is different: `LinearAttention.forward` explicitly builds `T × T` scores and a `T × T` decay mask so the algebra stays visible. It supports teacher forcing but does **not** realize linear-time or linear-memory training; that requires scan/chunk kernels outside this module.

In [ ]:
torch.manual_seed(0)
la = LinearAttention(128, 4)
x = torch.randn(1, 64, 128)

parallel = la(x)
state, outs = None, []
for t in range(64):
    out_t, state = la.step(x[:, t], state)
    outs.append(out_t)
recurrent = torch.stack(outs, dim=1)
print(f"max |parallel - recurrent| = {(parallel - recurrent).abs().max():.2e}")

In [ ]:
from statistics import median

from g2c.transformer import LayerKVCache


def sync_device():
    if device == "mps":
        torch.mps.synchronize()


def median_ms(fn, *, warmup=5, repeats=20):
    for _ in range(warmup):
        fn()
    sync_device()
    samples = []
    for _ in range(repeats):
        sync_device()
        start = time.perf_counter()
        fn()
        sync_device()
        samples.append((time.perf_counter() - start) * 1_000)
    return median(samples)


D, H = 128, 4
mha = MultiHeadAttention(D, H, causal=True).to(device)
la_bench = LinearAttention(D, H).to(device)
lengths = (64, 256, 1024, 4096, 16384)
full_times, linear_times = [], []
full_memory, linear_memory = [], []

with torch.no_grad():
    for T in lengths:
        keys = torch.randn(1, H, T, D // H, device=device)
        values = torch.randn_like(keys)
        x_full = torch.randn(1, 1, D, device=device)
        x_linear = x_full[:, 0]
        linear_state = la_bench.init_state(1, device)

        def full_step():
            cache = LayerKVCache(keys, values)
            return mha.forward_cached(x_full, cache)

        def linear_step():
            return la_bench.step(x_linear, linear_state)

        full_times.append(median_ms(full_step))
        linear_times.append(median_ms(linear_step))
        bytes_per_float = keys.element_size()
        full_memory.append(2 * T * D * bytes_per_float)
        d = D // H
        linear_memory.append((H * d * d + H * d) * bytes_per_float)

print(f"{'T':>6}  {'cached full ms':>15}  {'linear step ms':>15}  "
      f"{'full state MiB':>15}  {'linear state MiB':>17}")
for row in zip(lengths, full_times, linear_times, full_memory, linear_memory):
    T, full_ms, linear_ms, full_b, linear_b = row
    print(f"{T:>6}  {full_ms:>15.3f}  {linear_ms:>15.3f}  "
          f"{full_b / 2**20:>15.3f}  {linear_b / 2**20:>17.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lengths, full_times, marker="o", label="cached full")
axes[0].plot(lengths, linear_times, marker="o", label="linear step")
axes[0].set(xlabel="history tokens", ylabel="median ms / decoded token")
axes[0].legend()
axes[1].plot(lengths, [b / 2**20 for b in full_memory], marker="o", label="KV cache")
axes[1].plot(lengths, [b / 2**20 for b in linear_memory], marker="o", label="linear state")
axes[1].set(xlabel="history tokens", ylabel="state MiB")
axes[1].legend()
fig.tight_layout()
plt.show()

In [ ]:
"Question: What crossover, if any, did your cached-decoding benchmark show, and how noisy or monotonic were the timings? Explain why full attention's required scoring work and KV memory scale with history while the linear recurrent state stays fixed. Then explain why this result does NOT establish efficient linear-attention training in our implementation."
"Answer: "

## Exercise 2 — The three-way comparison

Train the StoryLM-1M architecture three ways on the 100MB TinyStories tier: all full attention, all linear attention, and `[linear, linear, linear, full]`. The projection and FFN sizes match; linear layers add only one learned decay scalar per head. Keep batches and step budgets identical, print the exact counts, and compare both validation loss and generated stories without presupposing a small gap.

In [ ]:
from g2c.artifacts import load_tokenized_corpus_artifact

CORPUS_NAME = "StoryLM-tinystories-100MB-v4096"
try:
    corpus = load_tokenized_corpus_artifact(CORPUS_NAME)
except FileNotFoundError as exc:
    raise RuntimeError(
        "TinyStories artifacts are missing. Run ./datasets.sh --tiny, "
        "then rerun this cell."
    ) from exc

pair = corpus.split(train_fraction=0.9, chunk_tokens=100_000, seed=12)
train_ids, val_ids = pair.train, pair.val
tokenizer = corpus.tokenizer
VOCAB = tokenizer.effective_vocab_size(4096)
print(f"corpus: {CORPUS_NAME}")
print(f"train tokens: {len(train_ids):,}; validation: {len(val_ids):,}")
print(f"effective vocabulary: {VOCAB:,}")

In [ ]:
from g2c.pretraining import get_lm_batch, lm_cross_entropy
from g2c.training import AdamW, clip_grad_norm_, cosine_with_warmup


def train_lm(model, *, steps, max_lr, min_lr=3e-5, warmup_steps=100,
             batch_size=16, context=128, weight_decay=0.05,
             grad_clip=1.0, log_every=100, eval_iters=5, seed=0):
    gen = torch.Generator().manual_seed(seed)
    model.to(device)
    opt = AdamW(model.parameters(), max_lr, weight_decay=weight_decay)
    history = {"step": [], "train_loss": [], "val_loss": []}
    for step in range(1, steps + 1):
        x, y = get_lm_batch(train_ids, batch_size, context, generator=gen)
        loss = lm_cross_entropy(model(x.to(device)), y.to(device))
        opt.zero_grad()
        loss.backward()
        clip_grad_norm_(model.parameters(), grad_clip)
        opt.lr = cosine_with_warmup(
            step - 1, warmup_steps=warmup_steps, max_steps=steps,
            max_lr=max_lr, min_lr=min_lr,
        )
        opt.step()
        if step % log_every == 0:
            with torch.no_grad():
                losses = []
                for _ in range(eval_iters):
                    vx, vy = get_lm_batch(
                        val_ids, batch_size, context, generator=gen
                    )
                    losses.append(lm_cross_entropy(
                        model(vx.to(device)), vy.to(device)
                    ).item())
            history["step"].append(step)
            history["train_loss"].append(float(loss.detach()))
            history["val_loss"].append(sum(losses) / len(losses))
            print(f"step {step:>4}  val loss {history['val_loss'][-1]:.3f}")
    return history


storylm_1m = dict(vocab_size=VOCAB, embedding_dim=128, num_heads=4,
                  max_seq_len=256, hidden_dim=512)
patterns = {
    "full": ["full"] * 4,
    "linear": ["linear"] * 4,
    "hybrid [L,L,L,F]": ["linear", "linear", "linear", "full"],
}
run = dict(steps=1_000, max_lr=3e-4, batch_size=16, context=128,
           log_every=100, seed=21)
models, histories = {}, {}
for name, pattern in patterns.items():
    torch.manual_seed(1)
    model = HybridTransformerLM(
        layer_pattern=pattern, **storylm_1m
    )
    models[name] = model
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()):,} params")
    histories[name] = train_lm(model, **run)

plt.figure(figsize=(7, 4))
for name, history in histories.items():
    plt.plot(history["step"], history["val_loss"], label=name)
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend()
plt.title("StoryLM-1M on TinyStories: near-matched parameters, iso-step")
plt.show()

In [ ]:
from g2c.sampling import generate


def print_story(label, model, seed):
    prompt = "Once upon a time"
    prompt_ids = torch.tensor(
        tokenizer.encode_with_vocab_size(prompt, VOCAB), dtype=torch.long
    )
    ids = generate(
        model, prompt_ids, 120, temperature=0.8, top_p=0.9,
        repetition_penalty=1.1,
        eos_id=tokenizer.special_to_id.get("<|endoftext|>"),
        generator=torch.Generator().manual_seed(seed),
    )
    print(f"--- {label}\n{tokenizer.decode(ids.tolist())}\n")


for name, model in models.items():
    print_story(name, model, seed=7)

In [ ]:
"Question: What did your StoryLM-1M validation curves and generated stories show? If the gaps were small, explain why average next-token loss can underweight rare exact-retrieval failures; if they were not small, identify what the result says about this linear layer or training scale."
"Answer: "

## Exercise 3 — The recall probe

A synthetic key-value task — `f=3;q=8;...;k=5` then `?q` — isolates exact retrieval of an arbitrary earlier pair. Train each four-layer architecture on the task itself, then score accuracy by retrieval distance. Do not assume that full attention is flat, linear attention decays, or the hybrid repairs a gap: optimization and toy-scale capacity can produce other curves, and those curves are the result to explain.

In [ ]:
import random
import string

N_PAIRS = 12
PROBE_CHARS = sorted(set(string.ascii_lowercase + string.digits + ";=?"))
probe_stoi = {c: i for i, c in enumerate(PROBE_CHARS)}
PROBE_VOCAB = len(PROBE_CHARS)


def probe_batch(batch_size, rng):
    rows, distances = [], []
    for _ in range(batch_size):
        keys = rng.sample(string.ascii_lowercase, N_PAIRS)
        vals = [rng.choice(string.digits) for _ in keys]
        q = rng.randrange(N_PAIRS)
        text = ";".join(f"{k}={v}" for k, v in zip(keys, vals))
        text += f"?{keys[q]}{vals[q]}"
        rows.append([probe_stoi[c] for c in text])
        distances.append(N_PAIRS - q)  # how far back the answer lives
    ids_b = torch.tensor(rows, dtype=torch.long)
    return ids_b[:, :-1], ids_b[:, 1:], torch.tensor(distances)


def train_and_score(pattern, *, steps=600, seed=3):
    rng = random.Random(seed)
    torch.manual_seed(seed)
    model = HybridTransformerLM(PROBE_VOCAB, 64, pattern, 4, 64).to(device)
    opt = AdamW(model.parameters(), 3e-3)
    for step in range(steps):
        x, y, _ = probe_batch(64, rng)
        loss = lm_cross_entropy(model(x.to(device)), y.to(device))
        opt.zero_grad()
        loss.backward()
        opt.step()
    # score: accuracy of the final position (the answer digit), by distance
    correct = torch.zeros(N_PAIRS + 1)
    total = torch.zeros(N_PAIRS + 1)
    with torch.no_grad():
        for _ in range(20):
            x, y, dist = probe_batch(128, rng)
            pred = model(x.to(device)).argmax(-1).cpu()[:, -1]
            hits = (pred == y[:, -1]).float()
            for d, h in zip(dist.tolist(), hits.tolist()):
                correct[d] += h
                total[d] += 1
    return correct / total.clamp(min=1)


probe_patterns = {
    "full": ["full"] * 4,
    "linear": ["linear"] * 4,
    "hybrid [L,L,L,F]": ["linear", "linear", "linear", "full"],
}
plt.figure(figsize=(7, 4))
for name, pattern in probe_patterns.items():
    print(f"--- training {name}")
    acc = train_and_score(pattern)
    print(f"accuracy by distance: {[round(v, 3) for v in acc[1:].tolist()]}")
    plt.plot(range(1, N_PAIRS + 1), acc[1:], marker="o", label=name)
plt.xlabel("retrieval distance (pairs back)")
plt.ylabel("answer accuracy")
plt.title("Exact recall vs distance")
plt.legend()
plt.show()

In [ ]:
"Question: Describe the recall curves you actually measured, including any result that contradicted the expected full/linear/hybrid ordering. What mechanism could explain the pattern, and what additional run or control would distinguish an architectural limit from an optimization failure?"
"Answer: "

## Exercise 4 — Inspect fixed per-head decay

Watch the state norm over a long sequence at three fixed decay settings, then inspect the values learned by the TinyStories models. Each `γ_h` is one scalar for an entire head: it applies the same forgetting rate to every token. An input-dependent gate instead computes a decay or write strength from the current token, allowing retention to depend on content.

In [ ]:
plt.figure(figsize=(7, 4))
xs = torch.randn(1, 512, 64)
for logit, label in [(50.0, "γ ≈ 1.0 (never forget)"),
                     (4.6, "γ ≈ 0.99 (initial value)"),
                     (0.0, "γ = 0.5 (aggressive)")]:
    torch.manual_seed(4)
    la_g = LinearAttention(64, 4)
    la_g.gamma_logit.data.fill_(logit)
    state, norms = None, []
    with torch.no_grad():
        for t in range(512):
            _, state = la_g.step(xs[:, t], state)
            norms.append(state[0].norm().item())
    plt.plot(norms, label=label)
plt.xlabel("tokens seen"); plt.ylabel("‖S‖")
plt.title("State norm under three decays")
plt.legend()
plt.show()

for model_name, model in models.items():
    learned = [
        block.attn.decay.detach().cpu().tolist()
        for block in model.blocks
        if isinstance(block.attn, LinearAttention)
    ]
    if learned:
        print(f"{model_name}: learned decay by linear layer = {learned}")

In [ ]:
"Question: How did decay affect the state norm, and where did the trained per-head values move from their ≈0.99 initialization? Explain one behavior a token-dependent gate could learn that these fixed per-head scalars cannot."
"Answer: "

## Exercise 5 — The one-million-token memory bill

Compute the inference state explicitly at `T=1,000,000`, `D=4096`, `H=32`, fp16. Include both `S` and `z` for a linear layer, then compare a four-layer all-full stack, the `[L,L,L,F]` hybrid, and a four-layer all-linear stack.

In [ ]:
"Question: At T=1,000,000, D=4096, H=32, fp16, calculate (a) one full layer's KV cache, 2·T·D values, and (b) one linear layer's S-plus-z state, H·(D/H)^2 + D values. Then total the state for four full layers, [L,L,L,F], and four linear layers. What does this arithmetic establish—and what does it not establish about quality or speed?"
"Answer: "

## Exercise 6 — StoryLM-5M rerun (optional)

Run the next cell only if you want the larger comparison. It keeps the same TinyStories corpus and moves to Module 10's six-layer StoryLM-5M dimensions. The hybrid places one full layer after its first three linear layers: `[L,L,L,F,L,L]`. These three runs are substantially longer than the required StoryLM-1M path.

In [ ]:
storylm_5m = dict(vocab_size=VOCAB, embedding_dim=256, num_heads=8,
                  max_seq_len=256, hidden_dim=1024)
run_5m = dict(steps=2_000, max_lr=3e-4, batch_size=8, context=128,
              warmup_steps=200, log_every=200, seed=51)
patterns_5m = {
    "full": ["full"] * 6,
    "linear": ["linear"] * 6,
    "hybrid [L,L,L,F,L,L]": ["linear", "linear", "linear",
                                  "full", "linear", "linear"],
}
models_5m, histories_5m = {}, {}
for name, pattern in patterns_5m.items():
    torch.manual_seed(5)
    model = HybridTransformerLM(
        layer_pattern=pattern, **storylm_5m
    )
    models_5m[name] = model
    print(f"--- {name}: {sum(p.numel() for p in model.parameters()):,} params")
    histories_5m[name] = train_lm(model, **run_5m)

plt.figure(figsize=(7, 4))
for name, history in histories_5m.items():
    plt.plot(history["step"], history["val_loss"], label=name)
plt.xlabel("step"); plt.ylabel("val loss"); plt.legend()
plt.title("StoryLM-5M optional rerun")
plt.show()
for name, model in models_5m.items():
    print_story(f"{name} 5M", model, seed=9)

When complete, ask a coding agent to grade your notebook. Partial work is fine: the agent should grade answered questions and implemented sections, then skip blank prompts.